In [ ]:
# General imports
from dotenv import load_dotenv
import os
import glob
from pathlib import Path
import gradio as gr

# Imports for LLM
from openai import OpenAI

# Imports for RAG
import tiktoken
import numpy
from langchain_openai import OpenAIEmbeddings
from langchain_chroma import Chroma
from langchain_huggingface import HuggingFaceEmbeddings
from langchain_community.document_loaders import DirectoryLoader, TextLoader
from langchain_text_splitters import RecursiveCharacterTextSplitter
from sklearn.manifold import TSNE 
import plotly.graph_objects as go

In [22]:
load_dotenv(override=True)

OPENAI_API_KEY = os.getenv("OPENAI_API_KEY")

if OPENAI_API_KEY:
    print(f"OpenAI API Key exists and begins {OPENAI_API_KEY[:8]}")
else:
    print("OpenAI API Key not set")

OpenAI API Key exists and begins sk-proj-


In [ ]:
# Use one of the GPT's auto regressive LLM (commonly refered as just LLM)
MODEL = "gpt-4.1-nano"
db_name = "vector_db"

openai = OpenAI()

In [ ]:
knowledge_base_path = "knowledge-base/**/*.md"

# Return a list of paths matching a pathname pattern.
files = glob.glob(knowledge_base_path, recursive=True)

print(f"Found {len(files)} in the Business Enngineering executive knowledge base")

Found 76 in the business enngineering executive knowledge base


In [ ]:
files

In [37]:
entire_knowledge_base = ""

for file_path in files:
    with open(file_path, 'r', encoding='utf-8') as f:
        entire_knowledge_base += f.read()
        entire_knowledge_base += "\n\n"

print(f"Total chars in the knowledge base: {len(entire_knowledge_base)}")

Total chars in the knowledge base: 304434


In [41]:
encoding = tiktoken.encoding_for_model(MODEL)
tokens = encoding.encode(entire_knowledge_base)
tokens_count = len(tokens)

print(f"Total token count for {MODEL} is: {tokens_count}")

Total token count for gpt-4.1-nano is: 63555


In [45]:
# Load everything in the knowledgebase using LangChain's loaders

BASE_FOLDER = "knowledge-base"

sub_folders = glob.glob(BASE_FOLDER + "/*")

documents = []

for folder in sub_folders:
    doc_type = os.path.basename(folder)
    print(doc_type)




products
contracts
company
employees


In [25]:
SYSTEM_PREFIX = """
You represent Insurellm, the Insurance Tech company.
You are an expert in answering questions about Insurellm; its employees and its products.
You are provided with additional context that might be relevant to the user's question.
Give brief, accurate answers. If you don't know the answer, say so.

Relevant context:
"""

In [26]:
import re
from collections.abc import Mapping

def get_relevant_context_pyhthonic(message: str, knowledge: Mapping[str, str]) -> list[str]:
    """Return knowledge entries whose keys appear as words in `message`.

    Matching is case-insensitive; each entry is returned at most once,
    in order of first appearance.
    """
    words = re.findall(r"[a-z]+", message.lower())
    unique_words = dict.fromkeys(words)          # ordered de-dupe
    return [knowledge[w] for w in unique_words if w in knowledge]

In [27]:
get_relevant_context_pyhthonic("Who is lancaster?", knowledge)

["# Avery Lancaster\n\n## Summary\n- **Date of Birth**: March 15, 1985\n- **Job Title**: Co-Founder & Chief Executive Officer (CEO)\n- **Location**: San Francisco, California\n- **Current Salary**: $225,000  \n\n## Insurellm Career Progression\n- **2015 - Present**: Co-Founder & CEO  \n  Avery Lancaster co-founded Insurellm in 2015 and has since guided the company to its current position as a leading Insurance Tech provider. Avery is known for her innovative leadership strategies and risk management expertise that have catapulted the company into the mainstream insurance market.  \n\n- **2013 - 2015**: Senior Product Manager at Innovate Insurance Solutions  \n  Before launching Insurellm, Avery was a leading Senior Product Manager at Innovate Insurance Solutions, where she developed groundbreaking insurance products aimed at the tech sector.  \n\n- **2010 - 2013**: Business Analyst at Edge Analytics  \n  Prior to joining Innovate, Avery worked as a Business Analyst, focusing on market 

In [28]:
def additional_context(message, knowledge):
    relevant_context = get_relevant_context_pyhthonic(message, knowledge)
    if not relevant_context:
        result = "There is no additional context relevant to the user's question."
    else:
        result = "The following additional context might be relevant in answering the user's question:\n\n"
        result += "\n\n".join(relevant_context)
    return result

In [ ]:
print(additional_context("Who is Alex Lancaster?", knowledge))

In [32]:
def chat(message, history):
    system_message = SYSTEM_PREFIX + additional_context(message, knowledge)
    messages = [{"role": "system", "content": system_message}] + history + [{"role": "user", "content": message}]
    response = openai.chat.completions.create(model=MODEL, messages=messages)
    return response.choices[0].message.content

In [ ]:
view = gr.ChatInterface(chat).launch(inbrowser=True)